# 06 · Curate evidence and connect the knowledge graph

**Goal:** inspect a small application profile, validate its relationships and export interoperable records without a graph database.

The local JSON is the editorial source. `data/catalog.jsonld` in the repository is a generated semantic view. Satellites are [SOSA](https://www.w3.org/TR/vocab-ssn/) platforms, selected instruments are sensors, curated products are [DCAT 3](https://www.w3.org/TR/vocab-dcat-3/) datasets, and evidence links use [PROV-O](https://www.w3.org/TR/prov-o/). A **capability** is not an actual observation; no `sosa:Observation` instances are fabricated.

[SWEET](https://github.com/ESIPFed/sweet), [NASA GCMD keywords](https://www.earthdata.nasa.gov/learn/find-data/idn/gcmd-keywords), [GeoSPARQL](https://www.ogc.org/standard/geosparql/), and [Workflow Run RO-Crate](https://www.researchobject.org/workflow-run-crate/profiles/workflow_run_crate/) are extension points. The pilot does not claim validated mappings, spatial reasoning, or a reproducible execution crate.

[GEOSS](https://old.earthobservations.org/geoss.php) supplies a system-of-systems discovery context. The [GEO Knowledge Hub](https://gkhub.earthobservations.org/doc/faq) can inform future reusable application packages. Neither is treated as the one ontology for every entity, and no live federation is required for the offline poster.

In [ ]:
from eo_guide import load_catalog
catalog = load_catalog()
ids = [s['id'] for s in catalog['satellites']]
assert len(ids) == len(set(ids))
for satellite in catalog['satellites']:
    assert satellite['verified'] and satellite['status'] in {'operational', 'retired', 'planned', 'unverified'}
    for source_id in satellite['sources']:
        assert source_id in catalog['sources']
    for sensor in satellite['sensors']:
        for source_id in sensor['sources']:
            assert source_id in catalog['sources']
print('Unique platform IDs, explicit statuses and source references verified.')

In [ ]:
# Export a small, explicit platform/sensor view using standard vocabulary terms.
import json
from pathlib import Path
base = 'https://jltobias.github.io/JupyterLite-EO-Satellite-Guide/id/'
mission = next(s for s in catalog['satellites'] if s['id'] == 'landsat-8')
graph = {
    '@context': {'sosa': 'http://www.w3.org/ns/sosa/', 'rdfs': 'http://www.w3.org/2000/01/rdf-schema#', 'label': 'rdfs:label'},
    '@graph': [{'@id': base + mission['id'], '@type': 'sosa:Platform', 'label': mission['name'],
                'sosa:hosts': [{'@id': base + mission['id'] + '/' + i['id']} for i in mission['sensors']]}]
}
graph['@graph'].extend({'@id': base + mission['id'] + '/' + i['id'], '@type': 'sosa:Sensor', 'label': i['name']} for i in mission['sensors'])
Path('my-platform-graph.jsonld').write_text(json.dumps(graph, indent=2), encoding='utf-8')
graph

## Add a satellite responsibly

1. Choose an official operator, mission or instrument source. Record its exact URL, title, publisher and access date.
2. Create a stable platform ID, explicit status, status qualifier and verification date. Do not substitute a design lifetime for a verified mission end date.
3. Record orbit, instruments, observed properties, native resolution/sampling, temporal definitions, coverage and products with citations by field group. Use `null` or an explanatory string for uncurated values.
4. Validate `data/catalog.json` against `data/catalog.schema.json` using `python scripts/validate.py`, then run `python scripts/build.py --execute` and browser checks.
5. Audit the source terms before copying an image or dataset. Linking a source and redistributing its content have different requirements.

Refresh changing status and access records at least each teaching term, and record what changed. The current snapshot is intentionally finite. Expand toward NOAA/JPSS, Metop, Sentinel-3/5P, CBERS, KOMPSAT, SAOCOM and additional national/commercial missions only after curation.

## Exercise

Explain the difference between a sensor's capability, an observation, a dataset and an analysis result. Draft one new record with at least one explicitly unknown field. Identify which semantic mappings you can support and which remain unverified.